In [3]:
import os
import json
import torch
import matplotlib.pyplot as plt

TOP3_dic = {
    "THEIA_E3": 
    {
        "1": {
            "name" : "T2-C3",
            "percent": 37},
        "2":  {
            "name" : "T4-C9",
            "percent": 30},
        "3": {
            "name" : "T4-C3",
            "percent": 12},
    },
    "THEIA_E5": {
        "1": {
            "name" : "T4-C4",
            "percent": 65,},
        "2":  {
            "name" :"T4-C5" ,
            "percent": 18,},
        "3":  {
            "name" :"T1-C1" ,
            "percent": 13},
    },

    "TRACE_E3": {
        "1": {
            "name" : "T2-C3",
            "percent": 32,},
        "2":  {
            "name" : "T3-C8",
            "percent": 30,},
        "3":  {
            "name" : "T2-C2",
            "percent": 16},
    },

 "TRACE_E5": {
        "1": {
            "name" : "T1-C1",
            "percent": 68,},
        "2":  {
            "name" : "T4-C1",
            "percent": 11,},
        "3":  {
            "name" : "T2-C2" ,
            "percent": 5,},
    },

}

RESULTS_ROOT = "./results"

DATASETS = [
    "THEIA_E3",
    "THEIA_E5",
    "TRACE_E3",
    "TRACE_E5",
]

METHODS = [
    "Flash",
    "Velox",
    "Orthrus",
]

In [ ]:


# ============================================================
# METRIC CALCULATION
# ============================================================

def calculate_metrics(results_path):

    data = torch.load(
        results_path,
        map_location="cpu"
    )

    tp = set()
    fp = set()
    tn = set()
    fn = set()

    for key, item in data.items():

        y_hat = item["y_hat"]
        y_true = item["y_true"]

        if y_hat == 1:

            if y_true == 1:
                tp.add(key)
            else:
                fp.add(key)

        else:

            if y_true == 1:
                fn.add(key)
            else:
                tn.add(key)

    tp = len(tp)
    fp = len(fp)
    tn = len(tn)
    fn = len(fn)

    eps = 1e-10

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)

    f1 = (
        2 * precision * recall /
        (precision + recall + eps)
    )

    return {
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# ============================================================
# FIND RESULT FILE
# ============================================================

def find_result_file(dataset, method, configuration):

    path = os.path.join(
        RESULTS_ROOT,
        dataset,
        method,
        configuration,
        "results",
        "results.pth"
    )

    if os.path.exists(path):
        return path

    return None


# ============================================================
# DETERMINE CONFIGURATION
# ============================================================

def get_configurations():

    return [
        "1",
        "2",
        "3",
    ]


# ============================================================
# GET REMOVAL PERCENTAGE
# ============================================================

def get_profile_data(dataset, configuration):

    return TOP3_dic[
            dataset
        ][configuration]



# ============================================================
# COLLECT ALL RESULTS
# ============================================================

def collect_results():

    output = {}

    for dataset in DATASETS:

        output[dataset] = {}

        for method in METHODS:

            output[dataset][method]= {}
            output[dataset][method]["profiles"] =[]

            for configuration in get_configurations():

                dirname = (
                        dataset.lower()
                        + "_"
                        + configuration
                    )

                result_path1 = find_result_file(
                    dataset,
                    method,
                    dirname
                )
                result_path2 = find_result_file(
                    dataset,
                    method,
                    dirname+"_random"
                )

                if result_path1 is None :
                    print(
                        "MISSING:",
                        dataset,
                        method,
                        dirname
                    )
                    continue
                if result_path2 is None :
                    print(
                        "MISSING:",
                        dataset,
                        method,
                        dirname+"_random"
                    )
                print(result_path1)
                metrics1 = calculate_metrics(
                    result_path1
                )
                metrics2 = calculate_metrics(result_path2)

                profile_data = get_profile_data(
                    dataset,
                    configuration
                )
                profile = {
                    "profile" : profile_data["name"],
                    
                    "removes_pct" : profile_data["percent"],
                    "profile_removal" :{
                        "tp" : metrics1["TP"],
                        "fp" :metrics1["FP"],
                    },
                    "matched_random" :{
                        "tp" :metrics2["TP"],
                        "fp" :metrics2["FP"],
                    },
                }
                output[dataset][method]["profiles"].append(profile)

    return output


# ============================================================
# SAVE JSON
# ============================================================

results = collect_results()

with open(
    "behavioral_composition_results.json",
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )

print(
    "\nSaved:",
    "behavioral_composition_results.json"
)


./results\THEIA_E3\Flash\theia_e3_1\results\results.pth
./results\THEIA_E3\Flash\theia_e3_2\results\results.pth
./results\THEIA_E3\Flash\theia_e3_3\results\results.pth
./results\THEIA_E3\Velox\theia_e3_1\results\results.pth
./results\THEIA_E3\Velox\theia_e3_2\results\results.pth
./results\THEIA_E3\Velox\theia_e3_3\results\results.pth
./results\THEIA_E3\Orthrus\theia_e3_1\results\results.pth
./results\THEIA_E3\Orthrus\theia_e3_2\results\results.pth
./results\THEIA_E3\Orthrus\theia_e3_3\results\results.pth
./results\THEIA_E5\Flash\theia_e5_1\results\results.pth
./results\THEIA_E5\Flash\theia_e5_2\results\results.pth
./results\THEIA_E5\Flash\theia_e5_3\results\results.pth
./results\THEIA_E5\Velox\theia_e5_1\results\results.pth
./results\THEIA_E5\Velox\theia_e5_2\results\results.pth
./results\THEIA_E5\Velox\theia_e5_3\results\results.pth
./results\THEIA_E5\Orthrus\theia_e5_1\results\results.pth
./results\THEIA_E5\Orthrus\theia_e5_2\results\results.pth
./results\THEIA_E5\Orthrus\theia_e5_3\

In [13]:
import json


INPUT_FILE = "behavioral_composition_results.json"
OUTPUT_FILE = "rq2_profile_vs_random_table.tex"


# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------

def fmt_int(x):
    if x is None:
        return "--"
    return f"{int(x):,}"


def fmt_pct(x):
    if x is None:
        return "--"
    return f"{x:.2f}\\%"


# ---------------------------------------------------------
# Load results
# ---------------------------------------------------------

with open(INPUT_FILE, "r") as f:
    results = json.load(f)


# ---------------------------------------------------------
# Build LaTeX
# ---------------------------------------------------------

lines = []

lines.append(r"\begin{table*}[t]")
lines.append(r"\centering")
lines.append(
    r"\caption{Comparison of dominant behavioral-profile removal and "
    r"volume-matched random removal. TP and FP denote the number of true "
    r"positives and false positives, respectively.}"
)
lines.append(r"\label{tab:profile_vs_random}")

lines.append(r"\small")
lines.append(r"\begin{tabular}{lll r rr rr}")
lines.append(r"\hline")

lines.append(
    r"\textbf{Dataset} & "
    r"\textbf{PIDS} & "
    r"\textbf{Profile} & "
    r"\textbf{Removed} & "
    r"\multicolumn{2}{c}{\textbf{Profile removal}} & "
    r"\multicolumn{2}{c}{\textbf{Matched random}} \\"
)

lines.append(
    r"\cline{5-6} \cline{7-8}"
)

lines.append(
    r" & & & & "
    r"\textbf{TP} & \textbf{FP} & "
    r"\textbf{TP} & \textbf{FP} \\"
)

lines.append(r"\hline")


# ---------------------------------------------------------
# Rows
# ---------------------------------------------------------

for dataset in DATASETS:

    dataset_data = results.get(dataset, {})
    count = 0 

    for pid in METHODS:
        print(pid)
        count +=1
        count2 = 0 

        pid_data = dataset_data.get(pid, {})

        profiles = pid_data.get("profiles", [])

        first_row_for_pid = True

        for profile in profiles:
            count2+=1

            profile_name = profile["profile"]
            removed_pct = profile["removes_pct"]

            profile_tp = profile["profile_removal"]["tp"]
            profile_fp = profile["profile_removal"]["fp"]

            random_tp = profile["matched_random"]["tp"]
            random_fp = profile["matched_random"]["fp"]

            # Avoid repeating dataset / PIDS names
            if count == 2 and count2 == 2 :
                dataset_cell = dataset.replace("_", "-")
            else:
                dataset_cell = ""
            if count2 == 2 :
                pid_cell = pid
            else :
                pid_cell = ''
            lines.append(
                f"{dataset_cell} & "
                f"{pid_cell} & "
                f"{profile_name} & "
                f"{fmt_pct(removed_pct)} & "
                f"{fmt_int(profile_tp)} & "
                f"{fmt_int(profile_fp)} & "
                f"{fmt_int(random_tp)} & "
                f"{fmt_int(random_fp)} \\\\"
            )
        lines.append(
    r"\cline{2-8}"
)
    lines.append(r"\hline")
    lines.append(r"\hline")


lines.append(r"\end{tabular}")
lines.append(r"\end{table*}")


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

with open(OUTPUT_FILE, "w") as f:
    f.write("\n".join(lines))


print(f"Saved: {OUTPUT_FILE}")

Flash
Velox
Orthrus
Flash
Velox
Orthrus
Flash
Velox
Orthrus
Flash
Velox
Orthrus
Saved: rq2_profile_vs_random_table.tex
